# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:
1. 5 (2,3)
np.arrange(6) создает одномерный массив от 0 до 5 .reshape(2,3) преобразует его в матрицу из двух строк и трех столбцов a[1, 2] элемент во второй строке и в третьем столбце
2. 1 8
argmaх позиция первого появление максимального элемента max значение максимального элемента 
3. 99
b не создал копию а поэтому b[0] отсылается на a[0]
4. 2 0.5
а>3 создает массив из булеанов True=1 False=0
5. (2, 3) 0.25
np.zeros((2, 3)) создаёт массив нулей размера 2 на 3 np.linspace(0, 1, 5) делит отрезок от 0 до 1 на 5 равноудалённых точек поэтому второй элеиент 0.25

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [3]:
# База
pos = np.argmax(amount)
print(f"Позиция: {pos}")
print(f"Сумма: {amount[pos]:,.0f} ₸ | Клиент: {cust[pos]} | Город: {city[pos]}")

paid_revenue = amount[paid].sum()
unpaid_sum = amount[~paid].sum()
print(f"Выручка оплаченных заказов: {paid_revenue:,.0f} ₸")
print(f"Сумма неоплаченных заказов: {unpaid_sum:,.0f} ₸")

shymkent_mask = (city == "Шымкент")
shymkent_count = shymkent_mask.sum()
shymkent_share = shymkent_mask.mean() * 100

print(f"Заказов в Шымкенте: {shymkent_count}")
print(f"Доля от всех заказов: {shymkent_share:.2f}%")


Позиция: 1001
Сумма: 9,999,990 ₸ | Клиент: TEST | Город: Алматы
Выручка оплаченных заказов: 101,909,360 ₸
Сумма неоплаченных заказов: 7,974,000 ₸
Заказов в Шымкенте: 804
Доля от всех заказов: 14.72%


Выводы для менеджера (База):
1. Позиция 1001, сумма 9 999 990 ₸ (клиент TEST, Алматы) — самый дорогой заказ в базе является тестовым, поэтому его нужно исключать из отчётов.
2. Выручка оплаченных заказов составила 101 909 360 ₸ (или 71 909 390 ₸ без TEST), а неоплаченных — 7 974 000 ₸, которые магазин теряет на отменах.
3. В Шымкенте 804 заказа (14.72% от всех заказов) — это третий по объёму заказов город для магазина.

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [4]:
# Стандарт
mask_all = paid & (amount > 50_000)
loss_all = amount[mask_all].sum() * 0.15

mask_clean = paid & (amount > 50_000) & (cust != "TEST")
loss_clean = amount[mask_clean].sum() * 0.15

print(f"Недополученная выручка: {loss_all:,.1f} ₸ (заказов: {mask_all.sum()})")
print(f"Недополученная выручка (без TEST):  {loss_clean:,.1f} ₸ (заказов: {mask_clean.sum()})")
print(f"Разница: в {loss_all / loss_clean:.2f} раза")

for c in np.unique(city):
    c_mask = (city == c)
    share_all = (days[c_mask] <= 2).mean() * 100
    share_valid = (days[c_mask & ~np.isnan(days)] <= 2).mean() * 100
    print(f"{c:10s} | от всех заказов: {share_all:5.2f}% | от заказов с известным сроком: {share_valid:5.2f}%")

#Test and unpaid orders must be excluded
clean_mask = paid & (cust != "TEST")
avg = amount[clean_mask].mean()
print("Средний чек:", round(avg), "₸")

Недополученная выручка: 5,702,422.5 ₸ (заказов: 127)
Недополученная выручка (без TEST):  1,202,427.0 ₸ (заказов: 124)
Разница: в 4.74 раза
Актобе     | от всех заказов: 62.22% | от заказов с известным сроком: 62.85%
Алматы     | от всех заказов: 64.53% | от заказов с известным сроком: 66.51%
Астана     | от всех заказов: 63.53% | от заказов с известным сроком: 65.41%
Атырау     | от всех заказов: 64.38% | от заказов с известным сроком: 66.67%
Караганда  | от всех заказов: 67.84% | от заказов с известным сроком: 71.01%
Шымкент    | от всех заказов: 29.85% | от заказов с известным сроком: 30.69%
Средний чек: 14672 ₸


Выводы для менеджера (Стандарт):
4. Реальные потери от скидки 15% составят 1 202 427 ₸ (со всеми заказами — 5 702 422.5 ₸). Ответы различаются в 4.74 раза, потому что у тестового клиента TEST есть 3 оплаченных заказа по 9 999 990 ₸ (почти 30 млн ₸), которые дают лишние 4.5 млн ₸ «скидки».
5. В Шымкенте доля быстрой доставки за 1–2 дня всего 29.85% (30.69% без пустых сроков), тогда как в остальных 5 городах она держится на уровне 62–68% — логистику в Шымкенте нужно проверить.
6. Две ошибки в коде AI: он посчитал средний чек вместе с неоплаченными заказами (~paid) и не убрал тестовые заказы клиента TEST. Реальный средний чек оплаченных заказов без TEST равен 14 672 ₸ вместо завышенных 20 114 ₸.

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [5]:
# Челлендж
import time

t0 = time.perf_counter()
total_for = 0.0
for i in range(len(amount)):
    if paid[i]:
        total_for += amount[i]
time_for = time.perf_counter() - t0

t0 = time.perf_counter()
total_np = amount[paid].sum()
time_np = time.perf_counter() - t0

print(f"Цикл for:    {time_for*1000:.3f} мс (сумма: {total_for:,.0f} ₸)")
print(f"Маска NumPy: {time_np*1000:.3f} мс (сумма: {total_np:,.0f} ₸)")
print(time_for / time_np)


np.random.seed(42)

buys_A = (np.random.rand(1000) < 0.05).sum()
buys_B = (np.random.rand(1000) < 0.06).sum()
print(f"1 эксперимент -> Группа A: {buys_A} покупок, Группа B: {buys_B} покупок")

np.random.seed(42)
sim_A = (np.random.rand(100, 1000) < 0.05).sum(axis=1)
sim_B = (np.random.rand(100, 1000) < 0.06).sum(axis=1)
b_wins = (sim_B > sim_A).sum()

print(f"Из 100 экспериментов группа B оказалась лучше A в {b_wins} случаях")



Цикл for:    4.988 мс (сумма: 101,909,360 ₸)
Маска NumPy: 0.252 мс (сумма: 101,909,360 ₸)
19.779143585744247
1 эксперимент -> Группа A: 54 покупок, Группа B: 58 покупок
Из 100 экспериментов группа B оказалась лучше A в 84 случаях


Выводы для менеджера (Челлендж):
7. Сумма через маску NumPy считается примерно в 20–300 раз быстрее обычного цикла for, что позволяет мгновенно считать метрики на больших таблицах.
8. В одном тесте группа A сделала 54 покупки, а группа B — 58 покупок; при 100 повторениях группа B выиграла в 84 случаях из 100. Это говорит о том, что выборки в 1 000 клиентов на группу недостаточно для надёжного вывода при росте конверсии с 5% до 6%: в 16% экспериментов случайный разброс скрывает преимущество группы B. Чтобы уверенно принимать решение, размер выборки нужно увеличить в несколько раз.